# Erschließung – lokaler Prototyp (Colab-Version)

Dieses Notebook startet die **echte Anwendung** – React-Frontend, Express-Backend, SQLite-Archiv,
lokale OCR (Tesseract) und .docx-Export – innerhalb der Colab-VM. Es handelt sich um keine
Neuimplementierung: der komplette Original-Quellcode ist unten als Archiv eingebettet und wird
1:1 entpackt und ausgeführt.

**Architektur (siehe auch `README.md` im Quellcode):**
- Frontend: React + Vite
- Backend: Node + Express (nur lokal in der VM, Port 3001)
- Datenbank: SQLite (`better-sqlite3`)
- OCR: Tesseract (`tesseract.js`, läuft komplett offline/WASM)
- .docx-Erzeugung/-Lesen: `docx` / `mammoth`
- Automatische Groberschließung: regelbasierter Parser (`src/lib/parser.js`) +
  Klassifikation (`src/lib/classify.js`) – Seitenmarken, Datumsangaben, Themen- und
  Autorenerkennung, alles ohne KI. Optional zusätzlich Personen-Erkennung über ein lokales
  LLM via Ollama – in Colab nicht verfügbar, die App fällt dafür automatisch und ohne Fehler
  auf "kein Modell erreichbar" zurück.

**Wichtig, bevor du loslegst:**
- Zellen der Reihe nach ausführen (Laufzeit → Alle Zellen ausführen).
- Die Colab-VM ist **flüchtig**: Alles was du im Archiv speicherst, geht beim Trennen der
  Laufzeit verloren, wenn du es nicht vorher herunterlädst (letzte Zelle unten macht das).


## 1. Quellcode laden

Der komplette Projekt-Quellcode (ohne `node_modules`, ohne bereits erzeugte Archiv-/DB-Dateien) ist als base64-kodiertes ZIP in der nächsten Zelle eingebettet.

In [ ]:
_SOURCE_ZIP_B64 = """...
AABQSwUGAAAAADMAMwC2EQAAgOMwAAAA"""


In [ ]:
import base64, zipfile, io, os

PROJECT_DIR = "/content/erschliessung-app"
os.makedirs(PROJECT_DIR, exist_ok=True)

zip_bytes = base64.b64decode(_SOURCE_ZIP_B64)
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    zf.extractall(PROJECT_DIR)

print(f"Entpackt nach: {PROJECT_DIR}")
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for fn in sorted(files)[:10]:
            print(f"{indent}  {fn}")


## 2. Node.js installieren

Colab-VMs bringen standardmäßig kein aktuelles Node.js mit. Node 20.x via NodeSource-Setup-Skript installieren (braucht laut `README.md` mindestens Node ≥ 18).

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node -v
!npm -v


## 3. Abhängigkeiten installieren

`better-sqlite3` wird hier gegen die Linux-VM neu kompiliert (Colab bringt einen funktionierenden C++-Build-Toolchain mit) – das läuft in der Regel unauffällig durch, dauert aber ein bis zwei Minuten.

In [ ]:
%cd /content/erschliessung-app
!npm install


## 4. (Optional) Regressionstests laufen lassen

Sanity-Check, dass der eingebettete Quellcode und die Abhängigkeiten korrekt sind – testet den Parser gegen vier echte Tagebuchband-Transkripte (`test/parser.test.js`).

In [ ]:
!npm test


## 5. Frontend bauen & Server starten

Einmalig `vite build` (erzeugt `dist/`), danach startet der Express-Server – er liefert API **und** das gebaute Frontend über denselben Port 3001 aus (kein zweiter Dev-Server/Proxy nötig, siehe `server/index.js`).

In [ ]:
import subprocess, time, urllib.request

!npm run build

server_proc = subprocess.Popen(
    ["node", "server/index.js"],
    cwd="/content/erschliessung-app",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:3001/api/vocab", timeout=1)
        print("Server läuft auf Port 3001.")
        break
    except Exception:
        time.sleep(0.5)
else:
    print("Server antwortet nicht — Log-Ausgabe:")
    print(server_proc.stdout.read())


## 6. App im Notebook öffnen

Zeigt die laufende App direkt eingebettet im Notebook an (Colabs Port-Forwarding-Proxy). Drag & Drop, Formular, Archiv, .docx-Speichern funktionieren wie im echten Browser.

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(3001, height=900)


Alternative: in einem eigenen Browser-Tab öffnen statt im Iframe:

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(3001)


## Einschränkungen in der Colab-Umgebung

- **Flüchtiger Speicher:** `data/archiv.db` und `archive/*.docx` liegen nur in der VM — beim
  Trennen der Laufzeit sind sie weg. Lade wichtige Ergebnisse über die letzte Zelle herunter.
- **Personen-Erkennung (Ollama):** kein lokales LLM in Colab verfügbar — der "Personen"-Abschnitt
  bleibt leer, die App sagt das offen (`modellVerfuegbar: false`), stürzt aber nicht ab. Die
  regelbasierte Groberschließung (Seiten, Datumsangaben, Themen, Autorenname) läuft unverändert.
- **".docx im Standardprogramm öffnen":** ruft in der App `open`/`xdg-open` auf — in der
  kopflosen VM ohne Desktop schlägt das fehl. Nutze stattdessen die Download-Zelle unten.
- **OCR (Foto/PDF):** tesseract.js läuft rein clientseitig, aber im Server-Kontext identisch
  wie lokal — funktioniert in Colab genauso wie auf dem eigenen Rechner.


## Ergebnisse herunterladen

Packt `data/` (SQLite-Archiv) und `archive/` (.docx-Dateien) zusammen und lädt sie als ZIP herunter, bevor die Laufzeit endet.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/erschliessung-ergebnisse", "zip", "/content/erschliessung-app", "data")
# archive/ zusätzlich anhängen
import zipfile, os
with zipfile.ZipFile("/content/erschliessung-ergebnisse.zip", "a") as zf:
    archive_dir = "/content/erschliessung-app/archive"
    if os.path.isdir(archive_dir):
        for fn in os.listdir(archive_dir):
            if fn.endswith(".docx"):
                zf.write(os.path.join(archive_dir, fn), f"archive/{fn}")

files.download("/content/erschliessung-ergebnisse.zip")
